In [1]:
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset"
CSV_PATH   = f"{DRIVE_BASE}/train_dataset_baru.csv"
OUTPUT_DIR = f"{DRIVE_BASE}/pa3_indobert_final_v2"

MODEL_NAME = "indobenchmark/indobert-base-p1"
NUM_LABELS = 4       
MAX_WEIGHT = 30.0     

print("Membaca Dataset")
df = pd.read_csv(CSV_PATH)

texts  = df['text_clean'].astype(str).tolist()
labels = df['final_level'].astype(int).tolist()

label_counts = Counter(labels)
level_names  = {0: "Normal/Positif", 1: "Perlu Pemantauan",
                2: "Perlu Perhatian", 3: "Krisis/Darurat"}
print(f"   Total data (Label 0,1,2,3): {len(texts)}")
for lvl in sorted(label_counts):
    print(f"   Level {lvl} ({level_names[lvl]}): {label_counts[lvl]} data")

print("\n Menghitung Class Weights")
total_samples = len(labels)
class_weights = []
for lvl in range(NUM_LABELS):
    count  = label_counts.get(lvl, 1)
    weight = total_samples / (NUM_LABELS * count)
    weight = min(weight, MAX_WEIGHT)
    class_weights.append(weight)
    print(f"   Level {lvl}: {count} data → bobot = {weight:.2f}")

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Weights: {class_weights_tensor.tolist()}")

print("\nMembagi Data (80:20)")
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels     
)
print(f"   Training : {len(train_texts)} data")
print(f"   Validasi : {len(val_texts)} data")

print("\nMemuat Model IndoBERT")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model     = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,          
    ignore_mismatched_sizes=True
)

print("Tokenisasi Teks")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(val_texts,   truncation=True, padding=True, max_length=128)

class IndoBERTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = IndoBERTDataset(train_encodings, train_labels)
val_dataset   = IndoBERTDataset(val_encodings,   val_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = class_weights_tensor.to(device)
weighted_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

class WeightedTrainer(Trainer):
    """Weighted CrossEntropyLoss agar Level 2 & 3 tidak diabaikan model."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss    = weighted_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir                  = '/content/results',
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    warmup_steps                = 300,
    weight_decay                = 0.01,
    logging_dir                 = '/content/logs',
    logging_steps               = 50,
    eval_strategy               = "epoch",   
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    fp16                        = True,
)


trainer = WeightedTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
)

print("\nMemulai Training (Weighted Loss")
print(f"   Device: {device}")
trainer.train()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nMenyimpan model ke: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n" + "="*55)
print("Selesai! Model tersimpan di Google Drive.")
print(f"   Folder: {OUTPUT_DIR}")
print("="*55)


Mounted at /content/drive
Membaca Dataset
   Total data (Label 0,1,2,3): 4793
   Level 0 (Normal/Positif): 1126 data
   Level 1 (Perlu Pemantauan): 3577 data
   Level 2 (Perlu Perhatian): 78 data
   Level 3 (Krisis/Darurat): 12 data

2. Menghitung Class Weights...
   Level 0: 1126 data → bobot = 1.06
   Level 1: 3577 data → bobot = 0.33
   Level 2: 78 data → bobot = 15.36
   Level 3: 12 data → bobot = 30.00
   Weights: [1.0641652345657349, 0.33498743176460266, 15.36217975616455, 30.0]

3. Membagi Data (80:20)
   Training : 3834 data
   Validasi : 959 data

4. Memuat Model IndoBERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


8. Menyimpan model ke: /content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset/pa3_indobert_final_v2


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Selesai! Model tersimpan di Google Drive.
   Folder: /content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset/pa3_indobert_final_v2
